# Conditional Workflow: Customer Review Reply with an LLM

The LLM reads the review's sentiment, and that decides which branch runs.

```
                           -> positive_response -------------------------> END
START -> find_sentiment -> 
                           -> run_diagnosis -> negative_response --------> END
```

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b')

In [2]:
# Structured output so the router gets a clean 'positive' / 'negative', not free text
class Sentiment(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(description='Sentiment of the review')


class Diagnosis(BaseModel):
    issue_type: str = Field(description='Category of issue, e.g. UX, performance, bug, support')
    tone: str = Field(description='Emotional tone of the customer, e.g. angry, frustrated, calm')
    urgency: Literal['low', 'medium', 'high'] = Field(description='How urgent the issue is')


sentiment_model = model.with_structured_output(Sentiment)
diagnosis_model = model.with_structured_output(Diagnosis)

In [3]:
class ReviewState(TypedDict):
    review: str
    sentiment: str
    diagnosis: dict
    response: str

In [4]:
def find_sentiment(state: ReviewState):
    result = sentiment_model.invoke(f"What is the sentiment of this review?\n{state['review']}")
    return {'sentiment': result.sentiment}


def positive_response(state: ReviewState):
    prompt = f"Write a short warm thank-you reply to this review, and ask them to share feedback on our website.\n{state['review']}"
    return {'response': model.invoke(prompt).content}


def run_diagnosis(state: ReviewState):
    result = diagnosis_model.invoke(f"Diagnose this negative review.\n{state['review']}")
    return {'diagnosis': result.model_dump()}


def negative_response(state: ReviewState):
    d = state['diagnosis']
    prompt = (f"You are a support assistant. The user had a '{d['issue_type']}' issue, "
              f"sounded '{d['tone']}' and marked urgency as '{d['urgency']}'. "
              f"Write a short, empathetic reply that helps resolve it.\n{state['review']}")
    return {'response': model.invoke(prompt).content}

In [5]:
def check_sentiment(state: ReviewState) -> Literal['positive_response', 'run_diagnosis']:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [6]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)

graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [7]:
result = workflow.invoke({'review': 'The app is super smooth and the new dark mode is lovely. Great work!'})
print('Sentiment:', result['sentiment'])
print(result['response'])

Sentiment: positive
Hi there! 🌟

Thank you so much for the kind words—glad to hear you’re enjoying the smooth experience and the new dark mode! We’re thrilled you think it’s great work. If you have any more thoughts or suggestions, feel free to drop them on our website. Your feedback helps us keep improving! 

Thanks again for your support!


In [8]:
result = workflow.invoke({'review': 'The app crashes every time I try to log in. I have a payment due today and this is useless!'})
print('Sentiment:', result['sentiment'])
print('Diagnosis:', result['diagnosis'])
print(result['response'])

Sentiment: negative
Diagnosis: {'issue_type': 'bug', 'tone': 'angry', 'urgency': 'high'}
I’m really sorry you’re running into this right now—let’s get you back on track quickly.

1. **Quick workaround**: Try logging in from a different browser (Chrome, Firefox, or Safari) or incognito mode.  
2. **Clear cache**: Sometimes old data blocks the login. Clear your browser cache or try a fresh private window.  
3. **Check for updates**: Make sure your app or browser is up‑to‑date.  
4. **Tell us what you see**: If you get an error message, copy the text or take a screenshot and send it to support@ourapp.com.  
5. **Temporary access**: If you need to make the payment today, let us know and we can issue a one‑time login link.

We’re on it—please let us know if the workaround works or if you still see an error. We’ll prioritize getting you in as soon as possible.
